In [19]:
import json
import os

import open_clip
import pandas as pd
import torch
from PIL import Image
from matplotlib import pyplot as plt
from torch.utils.data.dataloader import DataLoader

from homework_3.functions import parse_img, api_response_to_tensor

In [20]:
class ClipModel(torch.nn.Module):
    def __init__(self, model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        super().__init__()
        self.clip_backbone, _, self.preprocess = open_clip.create_model_and_transforms(model_name=model_name,
                                                                                       pretrained=pretrained)
        for p in self.clip_backbone.parameters():
            p.requires_grad = False
        self.clip_backbone.eval()
        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')

        self.class_labels = ["class1", "class2", "class3", "class4"]
        self.class_tokens = self.tokenizer(self.class_labels)
        self.class_features = self.clip_backbone.encode_text(self.class_tokens)
        if torch.cuda.is_available():
            self.class_features /= self.class_features.norm(dim=-1, keepdim=True)
            self.class_features = self.class_features.cuda()

        self.head = torch.nn.Linear(self.clip_backbone.visual.output_dim,
                                    self.clip_backbone.visual.output_dim)
        if torch.cuda.is_available():
            self.clip_backbone.cuda()
            self.head.cuda()

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            image_features = self.clip_backbone.encode_image(images)
        image_features = self.head(image_features)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        class_similarity = (100.0 * image_features @ self.class_features.T).softmax(dim=-1)
        return class_similarity


In [21]:
class ApiResponsesDataset(torch.utils.data.Dataset):
    def __init__(self, clip_preprocess, img_dir="../img", img_df=None, responses_cache="../cache/responses.json"):
        super().__init__()

        self.clip_preprocess = clip_preprocess
        self.responses_json = json.load(open(responses_cache, "r"))

        self.raw_images = [Image.open(f"{img_dir}/{p}") for p in os.listdir(img_dir)]
        self.raw_images.extend(img_df)

        self.data = []
        self.labels = []

        self._preprocess()

    def _preprocess(self):
        self.data = [self.clip_preprocess(raw_img) for raw_img in self.raw_images]

        try:
            for raw_img in self.raw_images:
                _, img_hash = parse_img(raw_img)
                response = self.responses_json[img_hash]
                label = api_response_to_tensor(response)
                self.labels.append(label)
        except KeyError:
            exit(f"Dataset integrity error. Not all images have responses.")

    def __len__(self):
        return len(self.raw_images)

    def __getitem__(self, index):
        return self.data[index], self.labels[index]

In [22]:
# TODO Add validation
def train_model(model, optimizer, criterion, train_data, num_epochs):
    losses = []
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        epoch_loss = 0.0
        for data, labels in train_data:
            if torch.cuda.is_available():
                data = data.cuda()
                labels = labels.cuda()
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(train_data))
        print(f"Train loss: {losses[-1]}")

    plt.plot(losses)
    plt.title("Loss curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

    return model



In [23]:
WEIGHTS_DIR = "../weights"
RANDOM_SEED = 161
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
NAME = f"{NUM_EPOCHS}_{BATCH_SIZE}"

torch.random.manual_seed(RANDOM_SEED)

splits = {'train': 'cifar100/train-00000-of-00001.parquet', 'test': 'cifar100/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/uoft-cs/cifar100/" + splits["train"]).img[:100]

print(len(df))

clip_model = ClipModel()

dataset = ApiResponsesDataset(clip_model.preprocess, img_df=df)
train_dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

adam_optimizer = torch.optim.Adam(clip_model.head.parameters(), lr=LEARNING_RATE)

crossEntropyLoss = torch.nn.CrossEntropyLoss()

trained_model = train_model(model=clip_model,
                            optimizer=adam_optimizer,
                            criterion=crossEntropyLoss,
                            train_data=train_dataloader,
                            num_epochs=NUM_EPOCHS)

torch.save(trained_model.head.state_dict(), f"{WEIGHTS_DIR}/{NAME}.pth")


100


TypeError: Unexpected type <class 'dict'>